# Visualize gfv2 fabric batches (`batch_size=500`)

Shows the KD-tree recursive bisection that `aggregate.batching.spatial_batch()` applies to the gfv2 fabric (~361k HRUs) at the default `batch_size=500`. Each batch is a spatially contiguous group of HRUs that gdptools processes together with its own cached weight CSV at `<project>/weights/<source>_batch<N>.csv`.

The partitioning is deterministic: same fabric → identical HRU membership per batch (only the integer `batch_id` labels would permute if the .gpkg row order changed, since labels are assigned in DFS order over the recursion).

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

from nhf_spatial_targets.aggregate.batching import spatial_batch


In [ ]:
FABRIC_PATH = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/"
    "gfv2_param_v2/gfv2/fabric/gfv2_nhru_merged.gpkg"
)
BATCH_SIZE = 10000


In [ ]:
t0 = perf_counter()
fabric = gpd.read_file(FABRIC_PATH)
print(f"Loaded fabric in {perf_counter() - t0:.1f}s")
print(f"  HRUs : {len(fabric):,}")
print(f"  CRS  : {fabric.crs}")
print(f"  bbox : {tuple(round(v, 1) for v in fabric.total_bounds)}")


In [ ]:
t0 = perf_counter()
batched = spatial_batch(fabric, batch_size=BATCH_SIZE)
print(f"Partitioned in {perf_counter() - t0:.1f}s")

sizes = batched.groupby("batch_id").size().to_numpy()
n_batches = len(sizes)
print(f"  n_batches : {n_batches}")
print(f"  min size  : {sizes.min()}")
print(f"  max size  : {sizes.max()}")
print(f"  mean size : {sizes.mean():.1f}")
print(f"  median    : {np.median(sizes):.0f}")


## Main figure: HRU centroids colored by batch

Plotting 361k polygons directly is slow; centroids give the same spatial story at scatter speed. Colors are assigned by a random permutation of `batch_id` modulo 20 so spatially-adjacent batches (which are *not* adjacent in `batch_id`, since the DFS traversal walks subtrees) end up with contrasting colors. Fixed seed for reproducibility.

In [ ]:
centroids = batched.geometry.centroid
rng = np.random.default_rng(seed=0)
perm = rng.permutation(n_batches)
color_idx = perm[batched["batch_id"].to_numpy()] % 20

fig, ax = plt.subplots(figsize=(14, 9))
ax.scatter(
    centroids.x,
    centroids.y,
    c=color_idx,
    cmap="tab20",
    s=0.5,
    marker=".",
    linewidths=0,
)
ax.set_aspect("equal")
ax.set_title(
    f"gfv2 fabric \u2014 {n_batches} spatial batches "
    f"(batch_size={BATCH_SIZE}, {len(batched):,} HRUs, CRS {fabric.crs.to_string()})"
)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.tight_layout()
plt.show()


## Batch-size distribution

The KD-tree targets `batch_size` but actual sizes vary because the recursion stops when a partition drops below `batch_size // 2` (= 250 for `batch_size=500`). Useful sanity check that no batch is pathologically tiny or huge.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sizes, bins=40, edgecolor="black", linewidth=0.5)
ax.axvline(BATCH_SIZE, color="red", linestyle="--", label=f"target ({BATCH_SIZE})")
ax.axvline(
    BATCH_SIZE // 2,
    color="orange",
    linestyle=":",
    label=f"min ({BATCH_SIZE // 2})",
)
ax.set_xlabel("HRUs per batch")
ax.set_ylabel("count")
ax.set_title("Batch size distribution")
ax.legend()
plt.tight_layout()
plt.show()
